In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# CONFIG
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
MAX_LEN, BATCH_SIZE, EPOCHS, LR = 128, 32, 3, 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DYNAMIC CHUNK LABEL EXTRACTION
def extract_chunk_labels(*paths):
    label_set = set()
    for path in paths:
        with open(path) as f:
            for line in f:
                if line.strip() == "":
                    continue
                splits = line.strip().split()
                label_set.add(splits[2])
    return sorted(label_set)

chunk_labels = extract_chunk_labels("Chunk_train.txt", "Chunk_test.txt")
chunk_label2id = {label: i for i, label in enumerate(chunk_labels)}
chunk_id2label = {i: label for label, i in chunk_label2id.items()}
print("Detected chunk labels:", chunk_labels)

# CHUNKING DATASET CLASS
class ChunkingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(self.texts[idx], is_split_into_words=True,
                                   return_offsets_mapping=True, padding='max_length',
                                   truncation=True, max_length=self.max_len)
        input_ids = torch.tensor(encodings['input_ids'])
        attention_mask = torch.tensor(encodings['attention_mask'])
        label_ids = torch.full((self.max_len,), -100)
        for i, w in enumerate(encodings.word_ids()):
            if w is not None and w < len(self.labels[idx]):
                label_ids[i] = self.labels[idx][w]
        return input_ids, attention_mask, label_ids

# DATA LOADING FUNCTIONS
def load_and_split_chunk_data(path, test_size=0.1):
    texts, labels = [], []
    with open(path) as f:
        tokens, tags = [], []
        for line in f:
            if line.strip() == "":
                if tokens:
                    texts.append(tokens)
                    labels.append([chunk_label2id[t] for t in tags])
                    tokens, tags = [], []
                continue
            splits = line.strip().split()
            tokens.append(splits[0])
            tags.append(splits[2])
    return train_test_split(texts, labels, test_size=test_size, random_state=42)

# LOAD DATA
chunk_train_texts, chunk_val_texts, chunk_train_labels, chunk_val_labels = load_and_split_chunk_data("Chunk_train.txt")

# LOAD TEST DATA
chunk_test_texts, chunk_test_labels = [], []
with open("Chunk_test.txt") as f:
    tokens, tags = [], []
    for line in f:
        if line.strip() == "":
            if tokens:
                chunk_test_texts.append(tokens)
                chunk_test_labels.append([chunk_label2id[t] for t in tags])
                tokens, tags = [], []
            continue
        splits = line.strip().split()
        tokens.append(splits[0])
        tags.append(splits[2])

# LOADERS
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(ChunkingDataset(chunk_train_texts, chunk_train_labels, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ChunkingDataset(chunk_val_texts, chunk_val_labels, tokenizer, MAX_LEN), batch_size=1)
test_loader = DataLoader(ChunkingDataset(chunk_test_texts, chunk_test_labels, tokenizer, MAX_LEN), batch_size=1)

# GENERATOR
class Generator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        logits = self.fc(output)
        return logits

# DISCRIMINATOR
class Discriminator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size + num_labels  # 312 + 22 = 334
        self.attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=2, batch_first=True)
        self.fc = nn.Linear(self.hidden_size, 2)

    def forward(self, input_ids, attention_mask, tag_logits):
        text_embeds = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        tag_embeds = F.gumbel_softmax(tag_logits, tau=0.5, hard=False)
        combined = torch.cat([text_embeds, tag_embeds], dim=-1)
        combined = combined[:, :attention_mask.size(1), :]
        attn_output, _ = self.attention(combined, combined, combined)
        logits = self.fc(attn_output)
        return logits

# MODELS + OPTIMIZERS
G = Generator(MODEL_NAME, len(chunk_label2id)).to(DEVICE)
D = Discriminator(MODEL_NAME, len(chunk_label2id)).to(DEVICE)
optimizer_G = AdamW(G.parameters(), lr=LR)
optimizer_D = AdamW(D.parameters(), lr=LR)

ce_loss = nn.CrossEntropyLoss(ignore_index=-100)

# TRAINING LOOP
for epoch in range(EPOCHS):
    G.train(); D.train()
    for input_ids, attention_mask, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        # Generator
        tag_logits = G(input_ids, attention_mask)
        g_loss = ce_loss(tag_logits.view(-1, len(chunk_label2id)), labels.view(-1))

        # Discriminator
        d_logits = D(input_ids, attention_mask, tag_logits)
        real_targets = (labels != -100).long()
        d_loss = ce_loss(d_logits.view(-1, 2), real_targets.view(-1))

        # Backward
        optimizer_G.zero_grad(); optimizer_D.zero_grad()
        (g_loss + d_loss).backward()
        optimizer_G.step(); optimizer_D.step()

    print(f"Epoch {epoch+1} | G Loss: {g_loss.item():.4f} | D Loss: {d_loss.item():.4f}")

# EVALUATION FUNCTION
def evaluate_generator(model, loader, name):
    print(f"\n--- {name} Chunking Evaluation ---")
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = labels.numpy()
            for p_seq, l_seq in zip(preds, labels):
                for p, l in zip(p_seq, l_seq):
                    if l != -100:
                        preds_all.append(chunk_id2label[p])
                        labels_all.append(chunk_id2label[l])
    print(classification_report(labels_all, preds_all, digits=4))

# FINAL VALIDATION + TEST EVALUATION
evaluate_generator(G, val_loader, "Validation")
evaluate_generator(G, test_loader, "Test")


Detected chunk labels: ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


Epoch 1: 100%|██████████| 101/101 [10:13<00:00,  6.08s/it]


Epoch 1 | G Loss: 0.9187 | D Loss: 0.0059


Epoch 2: 100%|██████████| 101/101 [09:54<00:00,  5.89s/it]


Epoch 2 | G Loss: 0.5936 | D Loss: 0.0019


Epoch 3: 100%|██████████| 101/101 [09:41<00:00,  5.76s/it]


Epoch 3 | G Loss: 0.4323 | D Loss: 0.0005

--- Validation Chunking Evaluation ---


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      B-ADJP     0.0000    0.0000    0.0000        81
      B-ADVP     0.6500    0.4509    0.5324       173
     B-CONJP     0.0000    0.0000    0.0000         3
      B-INTJ     0.0000    0.0000    0.0000         1
       B-LST     0.0000    0.0000    0.0000         1
        B-NP     0.9395    0.9290    0.9342      2691
        B-PP     0.8704    0.9876    0.9253       809
       B-PRT     0.0000    0.0000    0.0000        23
      B-SBAR     0.8750    0.3977    0.5469        88
        B-VP     0.9207    0.9237    0.9222       917
      I-ADJP     0.0000    0.0000    0.0000        16
      I-ADVP     0.0000    0.0000    0.0000        10
     I-CONJP     0.0000    0.0000    0.0000         3
        I-NP     0.9328    0.9520    0.9423      3149
        I-PP     0.0000    0.0000    0.0000        12
      I-SBAR     0.0000    0.0000    0.0000         3
        I-VP     0.8026    0.9217    0.8580       613
           O     0.9453    

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
